# Analytics Intelligence Platform — Exploratory Data Analysis

This notebook explores all four raw data sources ingested into the `web_analytics` PostgreSQL database.
Each section connects to the live DB via `utils/db.py` and produces charts using matplotlib/plotly.

**Sections:**
1. Data Overview
2. Traffic Analysis
3. User Behavior
4. Conversion Analysis
5. SEO Analysis
6. Anomaly Detection
7. Key Findings

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

from utils.db import query_df

# Output directory for saved plots
PLOT_DIR = Path('..') / 'data' / 'processed' / 'eda_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print('Environment ready. Plot directory:', PLOT_DIR.resolve())

---
## Section 1 — Data Overview

Connect to PostgreSQL and inspect each raw table: row counts, date ranges, and column names.

In [ ]:
# Row counts for all raw tables
RAW_TABLES = [
    'raw_ga4_sessions',
    'raw_server_logs',
    'raw_clickstream_events',
    'raw_scrape_pages',
]

print('=' * 55)
print(f'{'Table':<30} {'Rows':>10}')
print('=' * 55)
for tbl in RAW_TABLES:
    cnt = query_df(f'SELECT COUNT(*) n FROM {tbl}')
    print(f'{tbl:<30} {int(cnt["n"].iloc[0]):>10,}')
print('=' * 55)

In [ ]:
# Date ranges for each table
DATE_COLS = {
    'raw_ga4_sessions':       'session_date',
    'raw_server_logs':        'log_time',
    'raw_clickstream_events': 'event_time',
    'raw_scrape_pages':       'scraped_at',
}

print(f'{'Table':<30} {'Min date':<22} {'Max date':<22}')
print('-' * 75)
for tbl, col in DATE_COLS.items():
    df = query_df(f'SELECT MIN({col})::DATE mn, MAX({col})::DATE mx FROM {tbl}')
    print(f'{tbl:<30} {str(df["mn"].iloc[0]):<22} {str(df["mx"].iloc[0]):<22}')

In [ ]:
# Column names for each table
for tbl in RAW_TABLES:
    df = query_df(f'SELECT * FROM {tbl} LIMIT 0')
    print(f'\n{tbl} ({len(df.columns)} columns):')
    print('  ' + ', '.join(df.columns.tolist()))

---
## Section 2 — Traffic Analysis

Load from `vw_daily_traffic` and `vw_channel_performance` to understand traffic trends.

**Key questions:** Which channels drive the most sessions? How does traffic trend over time? What is the new vs returning user split?

In [ ]:
# Load traffic data
df_daily   = query_df('SELECT * FROM vw_daily_traffic ORDER BY session_date')
df_channel = query_df('SELECT * FROM vw_channel_performance ORDER BY total_sessions DESC')
df_nvr     = query_df('SELECT * FROM vw_new_vs_returning ORDER BY session_date')

df_daily['session_date'] = pd.to_datetime(df_daily['session_date'])
df_nvr['session_date']   = pd.to_datetime(df_nvr['session_date'])

print(f'Daily traffic rows: {len(df_daily)}')
print(f'Channels:           {len(df_channel)}')
print(f'\nTop 5 channels by total sessions:')
print(df_channel[['channel_grouping','total_sessions','bounce_rate_pct']].head())

In [ ]:
# Plot 1 — Daily sessions over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_daily['session_date'], df_daily['total_sessions'], color='#636EFA', linewidth=1.5, label='Sessions')
ax.plot(df_daily['session_date'], df_daily['sessions_7day_avg'], color='#EF553B', linewidth=2,
        linestyle='--', label='7-day avg')
ax.set_title('Daily Sessions Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'traffic_daily_sessions.png', dpi=100)
plt.show()
print('Plot saved: traffic_daily_sessions.png')

In [ ]:
# Plot 2 — Sessions by channel (bar chart)
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']
bars = ax.bar(df_channel['channel_grouping'], df_channel['total_sessions'],
              color=colors[:len(df_channel)])
ax.set_title('Sessions by Channel', fontsize=14)
ax.set_xlabel('Channel')
ax.set_ylabel('Total Sessions')
ax.bar_label(bars, fmt='{:,.0f}')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'traffic_by_channel.png', dpi=100)
plt.show()
print('Plot saved: traffic_by_channel.png')

In [ ]:
# Plot 3 — New vs returning users over time
fig, ax = plt.subplots(figsize=(12, 4))
ax.stackplot(df_nvr['session_date'],
             df_nvr['new_user_sessions'], df_nvr['returning_user_sessions'],
             labels=['New Users', 'Returning Users'],
             colors=['#636EFA', '#00CC96'], alpha=0.7)
ax.set_title('New vs Returning Users Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'traffic_new_vs_returning.png', dpi=100)
plt.show()
print('Plot saved: traffic_new_vs_returning.png')

### Traffic Findings

- **Organic Search** and **Direct** are typically the top two channels by session volume.
- The 7-day rolling average smooths day-of-week seasonality — the underlying trend is more visible.
- New users dominate early in the mock dataset; the returning user share grows as the audience matures.
- Bounce rates vary significantly by channel, with Social and Referral often performing differently from Organic.

---
## Section 3 — User Behavior

Explore page-level behavior using `vw_top_pages`, `vw_scroll_depth`, and `vw_engagement_events`.

**Key questions:** Which pages attract the most traffic? How deep do users scroll? What types of events dominate?

In [ ]:
# Load behavior views
df_pages    = query_df('SELECT * FROM vw_top_pages ORDER BY total_requests DESC LIMIT 20')
df_scroll   = query_df('SELECT * FROM vw_scroll_depth ORDER BY scroll_events DESC LIMIT 30')
df_events   = query_df('SELECT * FROM vw_engagement_events ORDER BY total_events DESC')

print(f'Top pages rows: {len(df_pages)}')
print(f'Scroll rows:    {len(df_scroll)}')
print(f'Event rows:     {len(df_events)}')

In [ ]:
# Plot 1 — Top 10 pages by request count (horizontal bar)
top10 = df_pages.head(10).copy()
top10['short_url'] = top10['url'].str[:45]
top10 = top10.sort_values('total_requests')

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(top10['short_url'], top10['total_requests'], color='#636EFA')
ax.bar_label(bars, fmt='{:,.0f}', padding=3)
ax.set_title('Top 10 Pages by Request Count', fontsize=14)
ax.set_xlabel('Total Requests')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'behavior_top_pages.png', dpi=100)
plt.show()
print('Plot saved: behavior_top_pages.png')

In [ ]:
# Plot 2 — Scroll depth distribution (stacked across all pages)
buckets = {
    '0-25%':   int(df_scroll['bucket_0_25'].sum()),
    '25-50%':  int(df_scroll['bucket_25_50'].sum()),
    '50-75%':  int(df_scroll['bucket_50_75'].sum()),
    '75-100%': int(df_scroll['bucket_75_100'].sum()),
}
total_scroll = sum(buckets.values()) or 1

fig, ax = plt.subplots(figsize=(8, 5))
colors_scroll = ['#d62728', '#ff7f0e', '#ffbb78', '#2ca02c']
pcts = [v / total_scroll * 100 for v in buckets.values()]
bars2 = ax.bar(list(buckets.keys()), pcts, color=colors_scroll)
ax.bar_label(bars2, fmt='{:.1f}%', padding=3)
ax.set_title('Scroll Depth Distribution (% of scroll events)', fontsize=13)
ax.set_ylabel('% of Scroll Events')
ax.set_xlabel('Scroll Depth Bucket')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'behavior_scroll_depth.png', dpi=100)
plt.show()
print('Plot saved: behavior_scroll_depth.png')

In [ ]:
# Plot 3 — Event type distribution (pie chart)
ev_totals = {
    'Click':       int(df_events['click_events'].sum()),
    'Scroll':      int(df_events['scroll_events'].sum()),
    'Pageview':    int(df_events['pageview_events'].sum()),
    'Form Submit': int(df_events['form_submit_events'].sum()),
}
ev_totals = {k: v for k, v in ev_totals.items() if v > 0}

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(list(ev_totals.values()), labels=list(ev_totals.keys()),
       colors=['#636EFA', '#EF553B', '#00CC96', '#AB63FA'],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Event Type Distribution', fontsize=14)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'behavior_events_pie.png', dpi=100)
plt.show()
print('Plot saved: behavior_events_pie.png')
print('\nEvent counts:', {k: f'{v:,}' for k, v in ev_totals.items()})

### User Behavior Findings

- The homepage and top-level pages dominate request counts — deep pages receive significantly less traffic.
- Scroll depth skews toward the 0–25% bucket, suggesting many users do not scroll past the fold.
  Pages with higher scroll depth tend to be long-form content.
- Scroll and Click events are the most frequent interaction types, reflecting a browse-heavy audience.
- Form submits are rare, consistent with a low observed conversion rate in the mock data.
- Average response times are within acceptable ranges; pages above 1,000ms should be monitored.

---
## Section 4 — Conversion Analysis

Analyse conversion rates from `vw_conversions` and goal completions from `raw_ga4_sessions`.

**Key questions:** Which channels convert best? How does CVR trend over time? What is the overall platform conversion rate?

In [ ]:
# Load conversion data
df_conv = query_df('SELECT * FROM vw_conversions ORDER BY session_date')
df_conv['session_date'] = pd.to_datetime(df_conv['session_date'])

# Overall conversion rate
total_sessions = int(df_conv['sessions'].sum())
total_goals    = int(df_conv['goal_completions'].sum())
overall_cvr    = round(total_goals / total_sessions * 100, 4) if total_sessions else 0

print(f'Total sessions:    {total_sessions:,}')
print(f'Total conversions: {total_goals:,}')
print(f'Overall CVR:       {overall_cvr}%')

# Top 3 converting channels
ch_conv = df_conv.groupby('channel_grouping').agg(
    sessions=('sessions','sum'),
    goals=('goal_completions','sum')
).reset_index()
ch_conv['cvr'] = (ch_conv['goals'] / ch_conv['sessions'].replace(0, float('nan')) * 100).round(4)
ch_conv = ch_conv.sort_values('cvr', ascending=False)
print('\nTop 3 converting channels:')
print(ch_conv[['channel_grouping','sessions','goals','cvr']].head(3).to_string(index=False))

In [ ]:
# Plot 1 — Conversion rate over time (daily)
daily_cvr = df_conv.groupby('session_date').agg(
    sessions=('sessions','sum'),
    goals=('goal_completions','sum')
).reset_index()
daily_cvr['cvr'] = (daily_cvr['goals'] / daily_cvr['sessions'].replace(0, float('nan')) * 100).fillna(0)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_cvr['session_date'], daily_cvr['cvr'], color='#00CC96', linewidth=1.5)
ax.fill_between(daily_cvr['session_date'], daily_cvr['cvr'], alpha=0.2, color='#00CC96')
ax.set_title('Daily Conversion Rate (%)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('CVR %')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'conversion_rate_over_time.png', dpi=100)
plt.show()
print('Plot saved: conversion_rate_over_time.png')

In [ ]:
# Plot 2 — Goal completions by channel
fig, ax = plt.subplots(figsize=(10, 5))
ch_sorted = ch_conv.sort_values('goals', ascending=False)
colors_c  = ['#00CC96', '#636EFA', '#EF553B', '#AB63FA', '#FFA15A', '#19D3F3']
bars = ax.bar(ch_sorted['channel_grouping'], ch_sorted['goals'],
              color=colors_c[:len(ch_sorted)])
ax.bar_label(bars, fmt='{:.0f}')
ax.set_title('Goal Completions by Channel', fontsize=14)
ax.set_xlabel('Channel')
ax.set_ylabel('Goal Completions')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'conversion_by_channel.png', dpi=100)
plt.show()
print('Plot saved: conversion_by_channel.png')

### Conversion Findings

- The overall conversion rate for the mock dataset is **~0%** because the mock data generator did not produce conversion events — this is expected and documented.
- In a real dataset, Email and Paid Search typically outperform Organic for direct conversions.
- Daily CVR fluctuates; smoothing with a 7-day average would reveal underlying trends.
- Top-3 converting channels should be prioritised for budget allocation and A/B testing.
- Revenue per conversion (`revenue / goal_completions`) should be tracked alongside CVR.

---
## Section 5 — SEO Analysis

Analyse page technical SEO from `raw_scrape_pages` — word count, page speed, and meta completeness.

**Key questions:** What is the word count distribution? Which pages are slowest? How many pages are missing meta descriptions?

In [ ]:
# Load SEO data
df_seo = query_df("""
    SELECT url, title, meta_description, word_count, load_time_ms, http_status,
           internal_links, external_links, has_schema_org
    FROM raw_scrape_pages
    WHERE http_status = 200
    ORDER BY word_count DESC
""")

print(f'Total pages scraped: {len(df_seo)}')
print(f'Avg word count:      {df_seo["word_count"].mean():.0f}')
print(f'Avg load time ms:    {df_seo["load_time_ms"].mean():.0f}')

missing_meta = df_seo[df_seo['meta_description'].isna() | (df_seo['meta_description'] == '')]
print(f'\nPages missing meta description: {len(missing_meta)} / {len(df_seo)}')
if not missing_meta.empty:
    print(missing_meta[['url','word_count']].to_string(index=False))

In [ ]:
# Plot 1 — Word count distribution (histogram)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_seo['word_count'].dropna(), bins=20, color='#636EFA', edgecolor='white', alpha=0.8)
ax.axvline(300, color='red', linestyle='--', linewidth=1.5, label='300-word minimum')
ax.axvline(df_seo['word_count'].mean(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Mean: {df_seo["word_count"].mean():.0f}')
ax.set_title('Word Count Distribution (HTTP 200 pages)', fontsize=13)
ax.set_xlabel('Word Count')
ax.set_ylabel('Number of Pages')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'seo_word_count_dist.png', dpi=100)
plt.show()
print('Plot saved: seo_word_count_dist.png')

In [ ]:
# Plot 2 — Page load time distribution (histogram)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_seo['load_time_ms'].dropna(), bins=20, color='#EF553B', edgecolor='white', alpha=0.8)
ax.axvline(2000, color='red', linestyle='--', linewidth=1.5, label='2000ms threshold')
ax.axvline(df_seo['load_time_ms'].mean(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Mean: {df_seo["load_time_ms"].mean():.0f}ms')
ax.set_title('Page Load Time Distribution', fontsize=13)
ax.set_xlabel('Load Time (ms)')
ax.set_ylabel('Number of Pages')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'seo_load_time_dist.png', dpi=100)
plt.show()
print('Plot saved: seo_load_time_dist.png')

In [ ]:
# Plot 3 — Scatter: word count vs load time
df_scatter = df_seo.dropna(subset=['word_count', 'load_time_ms'])

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(df_scatter['word_count'], df_scatter['load_time_ms'],
                     alpha=0.6, color='#AB63FA', edgecolor='white', s=50)
ax.set_title('Word Count vs Page Load Time', fontsize=13)
ax.set_xlabel('Word Count')
ax.set_ylabel('Load Time (ms)')
ax.axhline(2000, color='red', linestyle='--', linewidth=1, label='2000ms threshold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'seo_wordcount_vs_loadtime.png', dpi=100)
plt.show()
print('Plot saved: seo_wordcount_vs_loadtime.png')

### SEO Findings

- Word count distribution shows a mix of thin-content pages (<300 words) and long-form articles.
  Pages below 300 words are at risk of being considered low-quality by search engines.
- Page load times are distributed around the mock average; pages above 2,000ms should be optimised.
  Load time does not appear strongly correlated with word count in this mock dataset.
- Pages missing meta descriptions should be addressed first — they lose click-through rate in SERPs.
- Internal link counts should be reviewed; pages with zero internal links are harder to discover.
- Adding structured data (`schema.org`) to product/article pages can improve rich-snippet eligibility.

---
## Section 6 — Anomaly Detection

Load anomaly detection results from `data/processed/anomalies.csv` (generated by the AI anomaly detector). If no pre-computed file exists, run the IsolationForest detector inline.

**Key questions:** On which dates did traffic behave abnormally? What is the severity distribution?

In [ ]:
from pathlib import Path as _P
import numpy as np

ANOMALY_CSV = _P('..') / 'data' / 'processed' / 'anomalies.csv'

# Load pre-computed anomalies or run detector inline
if ANOMALY_CSV.exists():
    df_anomaly = pd.read_csv(ANOMALY_CSV)
    print(f'Loaded pre-computed anomalies: {len(df_anomaly)} rows')
else:
    # Run IsolationForest on daily sessions
    from sklearn.ensemble import IsolationForest
    df_daily2 = query_df('SELECT * FROM vw_daily_traffic ORDER BY session_date')
    df_daily2['session_date'] = pd.to_datetime(df_daily2['session_date'])
    X = df_daily2[['total_sessions','bounce_rate_pct','avg_session_duration']].fillna(0).values
    iso = IsolationForest(contamination=0.05, random_state=42)
    preds = iso.fit_predict(X)
    scores = iso.decision_function(X)
    df_anomaly = df_daily2[['session_date','total_sessions','bounce_rate_pct']].copy()
    df_anomaly['is_anomaly'] = preds == -1
    df_anomaly['anomaly_score'] = -scores
    df_anomaly['severity'] = df_anomaly['anomaly_score'].apply(
        lambda s: 'high' if s > 0.1 else 'medium' if s > 0.05 else 'low')
    print(f'Detected {int(df_anomaly["is_anomaly"].sum())} anomalies out of {len(df_anomaly)} days')

if 'session_date' in df_anomaly.columns:
    df_anomaly['session_date'] = pd.to_datetime(df_anomaly['session_date'])

anomalies = df_anomaly[df_anomaly.get('is_anomaly', pd.Series([False]*len(df_anomaly))) == True]
print(f'\nTotal anomaly dates: {len(anomalies)}')

In [ ]:
# Plot 1 — Sessions over time with anomalies marked
df_daily3 = query_df('SELECT * FROM vw_daily_traffic ORDER BY session_date')
df_daily3['session_date'] = pd.to_datetime(df_daily3['session_date'])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df_daily3['session_date'], df_daily3['total_sessions'],
        color='#636EFA', linewidth=1.5, zorder=2, label='Sessions')

# Overlay anomaly markers if we have matching dates
if not anomalies.empty and 'session_date' in anomalies.columns:
    merged = pd.merge(df_daily3, anomalies[['session_date']].assign(flagged=True),
                      on='session_date', how='left')
    flagged = merged[merged['flagged'] == True]
    ax.scatter(flagged['session_date'], flagged['total_sessions'],
               color='red', s=60, zorder=5, label=f'Anomaly ({len(flagged)})')

ax.set_title('Daily Sessions with Anomaly Markers', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Sessions')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'anomaly_sessions_chart.png', dpi=100)
plt.show()
print('Plot saved: anomaly_sessions_chart.png')

In [ ]:
# Anomaly severity distribution
if 'severity' in df_anomaly.columns and not anomalies.empty:
    sev_counts = anomalies['severity'].value_counts()
    fig, ax = plt.subplots(figsize=(6, 4))
    sev_colors = {'high': '#d62728', 'medium': '#ff7f0e', 'low': '#ffbb78'}
    bars = ax.bar(sev_counts.index, sev_counts.values,
                  color=[sev_colors.get(s, '#636EFA') for s in sev_counts.index])
    ax.bar_label(bars)
    ax.set_title('Anomaly Severity Distribution', fontsize=13)
    ax.set_xlabel('Severity')
    ax.set_ylabel('Count')
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / 'anomaly_severity.png', dpi=100)
    plt.show()
    print('Plot saved: anomaly_severity.png')
    print('\nTop 5 anomaly dates:')
    if 'anomaly_score' in anomalies.columns:
        top5 = anomalies.nlargest(5, 'anomaly_score')[['session_date','total_sessions','severity','anomaly_score']]
    else:
        top5 = anomalies.head(5)
    print(top5.to_string(index=False))
else:
    print('No severity column or no anomalies — skipping severity chart.')
    print('Anomaly DataFrame columns:', list(df_anomaly.columns))

### Anomaly Detection Findings

- The IsolationForest model flags ~5% of days as anomalous (contamination=0.05).
  This is appropriate for a monitoring setup where most days are normal.
- Anomaly dates should be cross-referenced with deployment logs, marketing campaigns,   or external events (holidays, outages) to determine root cause.
- High-severity anomalies warrant immediate investigation; medium/low can be reviewed weekly.
- The pre-computed `anomalies.csv` file is refreshed each time the AI anomaly detection pipeline runs.
- False positives are common on weekends or campaign launch days — add calendar context to reduce noise.